# Field evaluation and supporting analyses

This notebook runs the field evaluation and the supporting analyses for the cocoa bean detection–classification pipeline. All results are written as JSON/CSV/PNG files to `OUT_DIR`, and every section can be resumed: sections whose results already exist are skipped.

Sections:
- **0. Setup** – mounts Drive, loads data and model definitions (run first).
- **A. Shared-encoder diagnostic** – checks which Swin weights shared-encoder configurations should use at inference (see note below).
- **B. Lab per-sample predictions** – confusion matrices, accuracy/ECE/Brier and exact McNemar tests on the GT-crop test set.
- **C. Unified field evaluation** – 20 configurations on smartphone images with white and sack backgrounds; per-stratum F1, McNemar, bootstrap CIs, lab–field Spearman correlation and field calibration.
- **D. Anchor analysis** – DE-optimised anchor values, mean best-anchor IoU (standard vs DE) and DE convergence.
- **E. Per-class AP** – standard vs DE anchors, with bootstrap CI of ΔmAP.
- **F. Efficiency** – parameters, GFLOPs, per-stage latency and throughput (GPU and CPU).
- **G. Grad-CAM** – X4 vs Y4 on lab and field crops.
- **H. Dataset sufficiency** – learning curves at 10–75% of the training data.
- **I. 5-fold cross-validation** – X1, X4 and X6.
- **J. Summary** – lists and packages all output files.

### Note on shared-encoder configurations
In the shared-encoder configurations (A4–A8, Y1–Y6) the detector is trained first, and the classifier is then trained on the *same* Swin encoder, which updates it. As a result, the detector checkpoint and the classifier checkpoint store **two different versions of the Swin weights**. An earlier field-evaluation script loaded the classifier and then the detector, so the classifier's Swin weights were overwritten by the detector's. In the field, the shared classifier therefore ran on an encoder other than the one it was trained with, whereas its lab accuracy was measured with the correct encoder.

**Section A** tests whether this weight mismatch explains the gap between the decoupled (X) and shared (Y) configurations.

In [ ]:
# --- 0. Setup ---
from google.colab import drive
drive.mount('/content/drive')
!pip -q install torchmetrics faster-coco-eval 2>/dev/null | tail -1

import os, json, copy, shutil, gc, time, pickle, random, warnings, re
from collections import OrderedDict, Counter, defaultdict
import numpy as np, pandas as pd, cv2
import matplotlib; import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageOps
from tqdm.auto import tqdm
from scipy.stats import chi2, binomtest, spearmanr, entropy as scipy_entropy
from scipy.optimize import differential_evolution
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.utils.class_weight import compute_class_weight
from sklearn.calibration import CalibratedClassifierCV
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.ops as tv_ops
from torchvision.models.detection import RetinaNet
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.ops import FeaturePyramidNetwork, sigmoid_focal_loss
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import timm, xgboost as xgb

warnings.filterwarnings('ignore')
matplotlib.rcParams.update({'font.family': 'serif', 'font.serif': ['Times New Roman', 'DejaVu Serif'],
                            'font.size': 11, 'savefig.dpi': 300, 'savefig.bbox': 'tight'})
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '')

# --- Paths (same as the ablation notebooks) ---
BASE_DIR   = '/content/drive/MyDrive/UMK/Datasets/MCC-02-dataset'
CKPT_A     = f'{BASE_DIR}/checkpoints_final_ablation'     # chain A
CKPT_XY    = f'{BASE_DIR}/checkpoints_full_clahe'         # chains X and Y
TEST_IMG_DIR = f'{BASE_DIR}/testing'                      # 128 field images
OUT_DIR    = f'{BASE_DIR}/bab4_completion'
os.makedirs(OUT_DIR, exist_ok=True)
LOCAL = '/content'
CROP_DIR = f'{LOCAL}/crops_gt'                            # GT crops (shared by A/X/Y)

if not os.path.exists(f'{LOCAL}/annotations_coco.json'):
    shutil.copy2(f'{BASE_DIR}/annotations_coco.json', f'{LOCAL}/annotations_coco.json')
if not os.path.exists(f'{LOCAL}/images'):
    print('Copying images/ ...'); shutil.copytree(f'{BASE_DIR}/images', f'{LOCAL}/images')
if not os.path.exists(f'{LOCAL}/splits'):
    print('Copying splits/ ...'); shutil.copytree(f'{BASE_DIR}/splits', f'{LOCAL}/splits')
if not os.path.exists(CROP_DIR):
    print('Copying GT crops from Drive backup ...')
    shutil.copytree(f'{BASE_DIR}/cropped_splits_gt_based_backup', CROP_DIR)

with open(f'{LOCAL}/annotations_coco.json') as f: coco = json.load(f)
cat_map = {c['id']: c['name'] for c in coco['categories']}
sorted_cat_ids = sorted(cat_map)
safe_label_map = {cid: i for i, cid in enumerate(sorted_cat_ids)}
class_names = [cat_map[c] for c in sorted_cat_ids]
class_to_idx = {n: i for i, n in enumerate(class_names)}
NUM_CLASSES = len(class_names)
print('Classes:', class_names)
assert class_names == ['Broken', 'Unfermented', 'Fermented', 'Moldy'], 'Class order differs from the ablation notebooks'

def save_json(obj, name):
    p = f'{OUT_DIR}/{name}'
    with open(p, 'w') as f: json.dump(obj, f, indent=2, default=lambda o: o.item() if hasattr(o, 'item') else str(o))
    print('  [saved]', p); return p
def load_json(name):
    p = f'{OUT_DIR}/{name}'
    return json.load(open(p)) if os.path.exists(p) else None

In [ ]:
# --- 0b. Model definitions (same as the A/X/Y ablation notebooks) ---
IMG_SIZE = 224
NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
val_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), NORM])
train_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(),
                               transforms.RandomRotation(20), transforms.ToTensor(), NORM])

def apply_clahe_rgb(pil_img):
    n = np.array(pil_img)
    n = cv2.bilateralFilter(n, d=5, sigmaColor=50, sigmaSpace=50)
    lab = cv2.cvtColor(n, cv2.COLOR_RGB2LAB); l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(4, 4)).apply(l)
    return Image.fromarray(cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB))

def _bchw(x):
    if x.dim() == 4: return x if x.shape[1] in [96, 192, 384, 768] else x.permute(0, 3, 1, 2)
    B, L, C = x.shape; H = int(np.sqrt(L)); return x.permute(0, 2, 1).view(B, C, H, H)

def _swin():
    return timm.create_model('swin_tiny_patch4_window7_224', pretrained=True, features_only=True, out_indices=(0, 1, 2))

class BasicSwinCLF(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__(); self.swin = _swin(); self.gap = nn.AdaptiveAvgPool2d(1); self.classifier = nn.Linear(384, num_classes)
    def forward(self, x): return self.classifier(self.gap(_bchw(self.swin(x)[2])).flatten(1))
    def get_softmax(self, x): return F.softmax(self.forward(x), 1)

class SwinCNN_NoSAFM(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__(); self.swin = _swin()
        self.em0 = nn.Conv2d(96, 128, 1); self.em1 = nn.Conv2d(192, 128, 1); self.em2 = nn.Conv2d(384, 128, 1)
        self.dm2 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm1 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm0 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.fm0 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm1 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm2 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_agg = nn.Conv2d(384, 256, 1); self.gap = nn.AdaptiveAvgPool2d(1)
        self.fc_latent = nn.Linear(128 * 3 + 256, 384); self.classifier = nn.Linear(384, num_classes)
    def get_latent_features(self, x):
        f = self.swin(x)
        e0 = self.em0(_bchw(f[0])); e1 = self.em1(_bchw(f[1])); e2 = self.em2(_bchw(f[2]))
        d2 = self.dm2(e2)
        d1 = self.dm1(F.interpolate(d2, size=e1.shape[2:], mode='bilinear', align_corners=False) + e1)
        d0 = self.dm0(F.interpolate(d1, size=e0.shape[2:], mode='bilinear', align_corners=False) + e0)
        f2 = self.fm2(torch.cat([e2, d2], 1)); f1 = self.fm1(torch.cat([e1, d1], 1)); f0 = self.fm0(torch.cat([e0, d0], 1))
        fagg = self.fm_agg(torch.cat([f0, F.interpolate(f1, size=f0.shape[2:], mode='bilinear', align_corners=False),
                                      F.interpolate(f2, size=f0.shape[2:], mode='bilinear', align_corners=False)], 1))
        lat = torch.cat([self.gap(f0).flatten(1), self.gap(f1).flatten(1), self.gap(f2).flatten(1), self.gap(fagg).flatten(1)], 1)
        return F.relu(self.fc_latent(lat))
    def forward(self, x): return self.classifier(self.get_latent_features(x))
    def get_softmax(self, x): return F.softmax(self.forward(x), 1)

class AdaptiveSwinCNN_EMDMFM(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__(); self.swin = _swin()
        self.em_proj0 = nn.Conv2d(96, 128, 1); self.em_proj1 = nn.Conv2d(192, 128, 1); self.em_proj2 = nn.Conv2d(384, 128, 1)
        self.dm_conv2 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm_conv1 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.dm_conv0 = nn.Sequential(nn.Conv2d(128, 128, 3, padding=1), nn.GELU())
        self.fm_conv0 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_conv1 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_conv2 = nn.Sequential(nn.Conv2d(256, 128, 1), nn.GELU())
        self.fm_agg = nn.Conv2d(384, 256, 1); self.gap = nn.AdaptiveAvgPool2d(1)
        self.safm_attention = nn.MultiheadAttention(embed_dim=128, num_heads=8, batch_first=True)
        self.safm_norm = nn.LayerNorm(128); self.safm_dropout = nn.Dropout(0.1)
        self.fc_latent = nn.Linear(128 * 3 + 256, 384); self.classifier = nn.Linear(384, num_classes)
    def get_latent_features(self, x):
        f = self.swin(x)
        e0 = self.em_proj0(_bchw(f[0])); e1 = self.em_proj1(_bchw(f[1])); e2 = self.em_proj2(_bchw(f[2]))
        d2 = self.dm_conv2(e2)
        d1 = self.dm_conv1(F.interpolate(d2, size=e1.shape[2:], mode='bilinear', align_corners=False) + e1)
        d0 = self.dm_conv0(F.interpolate(d1, size=e0.shape[2:], mode='bilinear', align_corners=False) + e0)
        f2 = self.fm_conv2(torch.cat([e2, d2], 1)); f1 = self.fm_conv1(torch.cat([e1, d1], 1)); f0 = self.fm_conv0(torch.cat([e0, d0], 1))
        fagg = self.fm_agg(torch.cat([f0, F.interpolate(f1, size=f0.shape[2:], mode='bilinear', align_corners=False),
                                      F.interpolate(f2, size=f0.shape[2:], mode='bilinear', align_corners=False)], 1))
        tokens = torch.cat([self.gap(t).flatten(1).unsqueeze(1) for t in [f0, f1, f2]], 1)
        att, _ = self.safm_attention(tokens, tokens, tokens)
        lat = torch.cat([self.safm_norm(tokens + self.safm_dropout(att)).flatten(1), self.gap(fagg).flatten(1)], 1)
        return F.relu(self.fc_latent(lat))
    def forward(self, x): return self.classifier(self.get_latent_features(x))
    def get_softmax(self, x): return F.softmax(self.forward(x), 1)

class TunedRetinaNetHead(RetinaNetClassificationHead):
    def __init__(self, *a, focal_alpha=0.5, focal_gamma=2.0, **k):
        super().__init__(*a, **k); self.focal_alpha = focal_alpha; self.focal_gamma = focal_gamma
    def compute_loss(self, targets, head_outputs, matched_idxs):
        losses = []
        for tgt, cl_p, mi_p in zip(targets, head_outputs['cls_logits'], matched_idxs):
            fg = torch.where(mi_p >= 0)[0]; nf = fg.numel()
            gt_ = torch.zeros_like(cl_p); gt_[fg, tgt['labels'][mi_p[fg]]] = 1.0
            vi = mi_p != -2
            losses.append(sigmoid_focal_loss(cl_p[vi], gt_[vi], alpha=self.focal_alpha, gamma=self.focal_gamma, reduction='sum') / max(1, nf))
        return sum(losses) / len(targets)

class SwinFPNBackbone(nn.Module):
    """Backbone for both standalone and shared detectors. Parameter names are identical
    (backbone.swin/fpn/ds1/ds2), so shared checkpoints load into a separate Swin
    instance; the encoder weight variant is set explicitly when loading."""
    def __init__(self):
        super().__init__(); self.swin = _swin(); self.out_channels = 256
        self.fpn = FeaturePyramidNetwork([96, 192, 384], 256)
        self.ds1 = nn.Conv2d(256, 256, 3, stride=2, padding=1); self.ds2 = nn.Conv2d(256, 256, 3, stride=2, padding=1)
    def forward(self, x):
        sf = self.swin(x); fp = self.fpn(OrderedDict([(str(i), _bchw(sf[i])) for i in range(3)]))
        out = OrderedDict([('0', fp['0']), ('1', fp['1']), ('2', fp['2'])])
        out['3'] = self.ds1(out['2']); out['4'] = self.ds2(out['3']); return out

# --- Anchors (chain A and chains X/Y use different JSON files) ---
BASE_SIZES = [16, 24, 32, 48, 64]
STD_SCALES, STD_RATIOS = [1.0, 1.25, 1.5], [0.5, 1.0, 2.0]
def make_gen(scales, ratios, base=BASE_SIZES):
    return AnchorGenerator(sizes=tuple(tuple(b * s for s in scales) for b in base),
                           aspect_ratios=tuple(tuple(ratios) for _ in base))
ANCHOR_JSON_A = f'{CKPT_A}/DE_anchor_config_final.json'
ANCHOR_JSON_XY = f'{CKPT_XY}/A4_shared_swin_v2_anchor_config.json'
acfg_A = json.load(open(ANCHOR_JSON_A)); acfg_XY = json.load(open(ANCHOR_JSON_XY))
print('Anchor A :', {k: acfg_A.get(k) for k in ['opt_scales', 'opt_ratios', 'de_mean_max_iou']})
print('Anchor XY:', {k: acfg_XY.get(k) for k in ['opt_scales', 'opt_ratios', 'de_mean_max_iou']})
SAME_ANCHORS = np.allclose(acfg_A['opt_scales'], acfg_XY['opt_scales']) and np.allclose(acfg_A['opt_ratios'], acfg_XY['opt_ratios'])
print('Anchors A and X/Y are identical' if SAME_ANCHORS else 'Anchors A and X/Y differ')
STD_GEN = make_gen(STD_SCALES, STD_RATIOS)
DE_GEN_A = make_gen(acfg_A['opt_scales'], acfg_A['opt_ratios'], acfg_A.get('base_sizes', BASE_SIZES))
DE_GEN_XY = make_gen(acfg_XY['opt_scales'], acfg_XY['opt_ratios'], acfg_XY.get('base_sizes', BASE_SIZES))

def build_det(anchor_gen, score_thresh=0.05):
    det = RetinaNet(backbone=SwinFPNBackbone(), num_classes=NUM_CLASSES + 1, anchor_generator=anchor_gen,
                    min_size=224, max_size=224, score_thresh=score_thresh, nms_thresh=0.5,
                    fg_iou_thresh=0.4, bg_iou_thresh=0.3)
    det.head.classification_head = TunedRetinaNetHead(256, anchor_gen.num_anchors_per_location()[0], NUM_CLASSES + 1)
    return det.to(DEVICE)

def _state(path): d = torch.load(path, map_location=DEVICE); return d.get('model_state', d)
def swin_state_from(state, prefix):
    return {k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}

# --- Registry of the 20 configurations ---
# clf: (class, ckpt); det: (ckpt, anchor); xgb: pkl; shared: True/False
CFG = {
 'A1': dict(clf=('basic', f'{CKPT_A}/A1_final_clf.pth'), det=(f'{CKPT_A}/A1_final_det.pth', 'std'), clahe=False, shared=False),
 'A2': dict(clf=('basic', f'{CKPT_A}/A1_final_clf.pth'), det=(f'{CKPT_A}/A2_final_det.pth', 'deA'), clahe=False, shared=False),
 'A3': dict(clf=('basic', f'{CKPT_A}/A3_final_clf.pth'), det=(f'{CKPT_A}/A2_final_det.pth', 'deA'), clahe=True, shared=False),
 'A4': dict(clf=('basic', f'{CKPT_A}/A4_final_clf.pth'), det=(f'{CKPT_A}/A4_final_det.pth', 'deA'), clahe=True, shared=True),
 'A5': dict(clf=('nosafm', f'{CKPT_A}/A5_final_clf.pth'), det=(f'{CKPT_A}/A5_final_det.pth', 'deA'), clahe=True, shared=True),
 'A6': dict(clf=('adaptive', f'{CKPT_A}/A6_final_clf.pth'), det=(f'{CKPT_A}/A6_final_det.pth', 'deA'), clahe=True, shared=True),
 'A7': dict(clf=('adaptive', f'{CKPT_A}/A6_final_clf.pth'), det=(f'{CKPT_A}/A6_final_det.pth', 'deA'), clahe=True, shared=True, xgb=f'{CKPT_A}/A7_final_xgb_nocal.pkl'),
 'A8': dict(clf=('adaptive', f'{CKPT_A}/A6_final_clf.pth'), det=(f'{CKPT_A}/A6_final_det.pth', 'deA'), clahe=True, shared=True, xgb=f'{CKPT_A}/A8_final_xgb_calibrated.pkl'),
 'X1': dict(clf=('basic', f'{CKPT_XY}/X1_baseline_clf.pth'), det=(f'{CKPT_XY}/X1_baseline_det.pth', 'std'), clahe=False, shared=False),
 'X2': dict(clf=('basic', f'{CKPT_XY}/X1_baseline_clf.pth'), det=(f'{CKPT_XY}/X2_de_anchor_det.pth', 'deXY'), clahe=False, shared=False),
 'X3': dict(clf=('nosafm', f'{CKPT_XY}/X3_emdmfm_nosafm_clf.pth'), det=(f'{CKPT_XY}/X2_de_anchor_det.pth', 'deXY'), clahe=False, shared=False),
 'X4': dict(clf=('adaptive', f'{CKPT_XY}/X4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/X2_de_anchor_det.pth', 'deXY'), clahe=False, shared=False),
 'X5': dict(clf=('adaptive', f'{CKPT_XY}/X4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/X2_de_anchor_det.pth', 'deXY'), clahe=False, shared=False, xgb=f'{CKPT_XY}/X5_xgb_nocal.pkl'),
 'X6': dict(clf=('adaptive', f'{CKPT_XY}/X4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/X2_de_anchor_det.pth', 'deXY'), clahe=False, shared=False, xgb=f'{CKPT_XY}/X6_xgb_calibrated_isotonic.pkl'),
 'Y1': dict(clf=('basic', f'{CKPT_XY}/Y1_baseline_clf.pth'), det=(f'{CKPT_XY}/Y1_baseline_det.pth', 'std'), clahe=False, shared=True),
 'Y2': dict(clf=('basic', f'{CKPT_XY}/Y2_de_anchor_clf.pth'), det=(f'{CKPT_XY}/Y2_de_anchor_det.pth', 'deXY'), clahe=False, shared=True),
 'Y3': dict(clf=('nosafm', f'{CKPT_XY}/Y3_emdmfm_nosafm_clf.pth'), det=(f'{CKPT_XY}/Y3_emdmfm_nosafm_det.pth', 'deXY'), clahe=False, shared=True),
 'Y4': dict(clf=('adaptive', f'{CKPT_XY}/Y4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/Y4_adaptiveswin_full_det.pth', 'deXY'), clahe=False, shared=True),
 'Y5': dict(clf=('adaptive', f'{CKPT_XY}/Y4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/Y4_adaptiveswin_full_det.pth', 'deXY'), clahe=False, shared=True, xgb=f'{CKPT_XY}/Y5_xgb_nocal.pkl'),
 'Y6': dict(clf=('adaptive', f'{CKPT_XY}/Y4_adaptiveswin_full_clf.pth'), det=(f'{CKPT_XY}/Y4_adaptiveswin_full_det.pth', 'deXY'), clahe=False, shared=True, xgb=f'{CKPT_XY}/Y6_xgb_calibrated_isotonic.pkl'),
}
GENS = {'std': STD_GEN, 'deA': DE_GEN_A, 'deXY': DE_GEN_XY}
CLF_CLS = {'basic': BasicSwinCLF, 'nosafm': SwinCNN_NoSAFM, 'adaptive': AdaptiveSwinCNN_EMDMFM}
missing = [p for c in CFG.values() for p in [c['clf'][1], c['det'][0], c.get('xgb')] if p and not os.path.exists(p)]
print('All checkpoints found' if not missing else f'Missing checkpoints: {missing}')

_cache = {}
def load_clf(cfg_id, swin='clf'):
    """swin='clf': Swin weights from the classifier checkpoint (correct weights for the classifier).
       swin='det': Swin weights from the detector checkpoint (as in the earlier field script)."""
    c = CFG[cfg_id]; key = ('clf', c['clf'][1], swin if c['shared'] else 'clf')
    if key in _cache: return _cache[key]
    m = CLF_CLS[c['clf'][0]](NUM_CLASSES).to(DEVICE)
    m.load_state_dict(_state(c['clf'][1]), strict=True)
    if c['shared'] and swin == 'det':
        m.swin.load_state_dict(swin_state_from(_state(c['det'][0]), 'backbone.swin.'), strict=True)
    m.eval(); _cache[key] = m; return m

def load_det(cfg_id, swin='det', score_thresh=0.05):
    """swin='det': Swin weights from the detector checkpoint (as used for the lab mAP).
       swin='clf': Swin weights from the classifier checkpoint (final shared-encoder state after sequential training)."""
    c = CFG[cfg_id]; key = ('det', c['det'][0], swin if c['shared'] else 'det', score_thresh)
    if key in _cache: return _cache[key]
    d = build_det(GENS[c['det'][1]], score_thresh)
    d.load_state_dict(_state(c['det'][0]), strict=True)
    if c['shared'] and swin == 'clf':
        d.backbone.swin.load_state_dict(swin_state_from(_state(c['clf'][1]), 'swin.'), strict=True)
    d.eval(); _cache[key] = d; return d

def load_xgb(cfg_id):
    p = CFG[cfg_id].get('xgb')
    if not p: return None
    if p not in _cache: _cache[p] = pickle.load(open(p, 'rb'))
    return _cache[p]

def free_cache():
    _cache.clear(); gc.collect(); torch.cuda.empty_cache()

# --- GT-crop dataset ---
def crop_df(split, root=CROP_DIR):
    rows = []
    for cls in class_names:
        d = f'{root}/{split}/{cls}'
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(('.jpg', '.png', '.jpeg')):
                rows.append({'image_path': f'{d}/{fn}', 'label': class_to_idx[cls],
                             'group': re.sub(r'^crop_|_\d+\.jpg$', '', fn)})
    return pd.DataFrame(rows)

class ClfDataset(Dataset):
    def __init__(self, df, tf, use_clahe=False): self.df = df.reset_index(drop=True); self.tf = tf; self.use_clahe = use_clahe
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]; img = ImageOps.exif_transpose(Image.open(r['image_path'])).convert('RGB')
        if self.use_clahe: img = apply_clahe_rgb(img)
        return self.tf(img), int(r['label']), r['image_path']

def loader(df, tf=val_tf, clahe=False, shuffle=False, bs=32):
    return DataLoader(ClfDataset(df, tf, clahe), batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)

TEST_DF = crop_df('test'); VAL_DF = crop_df('val'); TRAIN_DF = crop_df('train')
print(f'Crops: train={len(TRAIN_DF)} val={len(VAL_DF)} test={len(TEST_DF)}')

# --- Statistics ---
def mcnemar(correct_a, correct_b):
    a = np.asarray(correct_a, bool); b_ = np.asarray(correct_b, bool)
    b = int((a & ~b_).sum()); c = int((~a & b_).sum())
    if b + c == 0: return dict(b=b, c=c, chi2=0.0, p_cc=1.0, p_exact=1.0, odds_ratio=None, cohen_g=0.0)
    x = (abs(b - c) - 1) ** 2 / (b + c)
    return dict(b=b, c=c, chi2=round(x, 3), p_cc=float(1 - chi2.cdf(x, 1)),
                p_exact=float(binomtest(b, b + c, 0.5).pvalue),
                odds_ratio=(round(b / c, 3) if c else float('inf')), cohen_g=round(abs(b / (b + c) - 0.5), 3))

def ece_score(y, proba, n_bins=15):
    y = np.asarray(y); proba = np.asarray(proba); conf = proba.max(1); pred = proba.argmax(1)
    acc = (pred == y).astype(float); bins = np.linspace(0, 1, n_bins + 1); e = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1]) if i else (conf >= 0) & (conf <= bins[1])
        if m.sum(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
    return float(e)

def brier(y, proba):
    y = np.asarray(y); proba = np.asarray(proba); oh = np.eye(proba.shape[1])[y]
    return float(((proba - oh) ** 2).sum(1).mean())

def f1_from(tp, fp, fn):
    p = tp / (tp + fp) if tp + fp else 0.0; r = tp / (tp + fn) if tp + fn else 0.0
    return (2 * p * r / (p + r) if p + r else 0.0), p, r
print('Setup complete')

## A. Shared-encoder diagnostic
For each shared configuration (A4, A5, A6, Y1–Y4) this section computes:
1. classifier lab accuracy with the Swin weights from the **classifier** checkpoint vs from the **detector** checkpoint (as used by the earlier field script);
2. detector lab mAP with the **detector** Swin weights vs the **classifier** Swin weights (the final state of the shared encoder).

If `acc_with_det_swin` is much lower than `acc_with_clf_swin`, the field failure of the shared configurations is mainly caused by a weight mismatch at inference rather than by domain shift.

In [ ]:
# --- A. Shared-encoder diagnostic ---
def lab_predict(model, df, clahe, xgb_model=None, bs=32, desc=''):
    ys, probs, paths = [], [], []
    with torch.no_grad():
        for x, y, p in tqdm(loader(df, clahe=clahe, bs=bs), desc=desc, leave=False):
            x = x.to(DEVICE)
            if xgb_model is not None: pr = xgb_model.predict_proba(model.get_latent_features(x).cpu().numpy())
            else: pr = model.get_softmax(x).cpu().numpy()
            probs.append(pr); ys.extend(y.numpy()); paths.extend(p)
    return np.array(ys), np.vstack(probs), paths

def det_dataset_test():
    test_files = set()
    for cls in class_names:
        d = f'{LOCAL}/splits/test/{cls}'
        if os.path.exists(d): test_files |= set(os.listdir(d))
    imgs = [im for im in coco['images'] if im['file_name'].split('/')[-1] in test_files]
    anns = defaultdict(list)
    for a in coco['annotations']: anns[a['image_id']].append(a)
    return imgs, anns

DET_TEST_IMGS, DET_ANNS = det_dataset_test()
print('Detection test images:', len(DET_TEST_IMGS))

def det_predict_all(det, bs=16, desc=''):
    """Detector predictions on all test images (kept for bootstrapping)."""
    preds, tgts = [], []
    with torch.no_grad():
        for i in tqdm(range(0, len(DET_TEST_IMGS), bs), desc=desc, leave=False):
            batch = DET_TEST_IMGS[i:i + bs]; xs = []
            for info in batch:
                img = ImageOps.exif_transpose(Image.open(f"{LOCAL}/{info['file_name']}")).convert('RGB')
                xs.append(transforms.ToTensor()(img).to(DEVICE))
                an = DET_ANNS[info['id']]
                tgts.append({'boxes': torch.tensor([[a['bbox'][0], a['bbox'][1], a['bbox'][0] + a['bbox'][2], a['bbox'][1] + a['bbox'][3]] for a in an], dtype=torch.float32).reshape(-1, 4),
                             'labels': torch.tensor([safe_label_map[a['category_id']] + 1 for a in an], dtype=torch.int64)})
            for o in det(xs):
                preds.append({k: o[k].cpu() for k in ['boxes', 'scores', 'labels']})
    return preds, tgts

def map_of(preds, tgts, idx=None, per_class=False, iou=None):
    kw = dict(box_format='xyxy', iou_type='bbox', class_metrics=per_class)
    if iou is not None: kw['iou_thresholds'] = iou
    try: m = MeanAveragePrecision(backend='faster_coco_eval', **kw)
    except Exception: m = MeanAveragePrecision(**kw)
    idx = range(len(preds)) if idx is None else idx
    m.update([preds[i] for i in idx], [tgts[i] for i in idx]); return m.compute()

res_A = load_json('A_shared_diagnostic.json') or {}
for cid in ['A4', 'A5', 'A6', 'Y1', 'Y2', 'Y3', 'Y4']:
    if cid in res_A: print(cid, 'already done -> skip'); continue
    c = CFG[cid]; r = {}
    for sw in ['clf', 'det']:
        y, pr, _ = lab_predict(load_clf(cid, sw), TEST_DF, c['clahe'], desc=f'{cid} clf/{sw}-swin')
        r[f'acc_with_{sw}_swin'] = float((pr.argmax(1) == y).mean())
    for sw in ['det', 'clf']:
        p, t = det_predict_all(load_det(cid, sw), desc=f'{cid} det/{sw}-swin')
        m = map_of(p, t); r[f'map5095_with_{sw}_swin'] = float(m['map']); r[f'map50_with_{sw}_swin'] = float(m['map_50'])
    res_A[cid] = r; print(cid, r); save_json(res_A, 'A_shared_diagnostic.json'); free_cache()

dfA = pd.DataFrame(res_A).T
dfA['acc_drop_pp'] = (dfA['acc_with_clf_swin'] - dfA['acc_with_det_swin']) * 100
dfA['map5095_drop_pp'] = (dfA['map5095_with_det_swin'] - dfA['map5095_with_clf_swin']) * 100
display(dfA.round(4))
print('\nInterpretation:')
for cid, r in dfA.iterrows():
    flag = 'LARGE -> weight-loading artefact' if r['acc_drop_pp'] > 2 else 'small'
    print(f"  {cid}: lab accuracy drops {r['acc_drop_pp']:.2f} pp with detector Swin weights ({flag}); "
          f"mAP@50:95 drops {r['map5095_drop_pp']:.2f} pp when the detector uses classifier Swin weights")

## B. Lab per-sample predictions, confusion matrices and exact McNemar tests
All 18 distinct classifiers are re-evaluated on the 2,405 GT test crops (Swin weights from the classifier checkpoint). Per-sample predictions are saved to CSV for the exact McNemar tests and confusion matrices.

In [ ]:
# --- B. Lab per-sample predictions, confusion matrices, McNemar ---
LAB_CSV = f'{OUT_DIR}/B_lab_predictions.csv'
DISTINCT = ['A1', 'A3', 'A4', 'A5', 'A6', 'A7', 'A8', 'X1', 'X3', 'X4', 'X5', 'X6', 'Y1', 'Y2', 'Y3', 'Y4', 'Y5', 'Y6']
lab = pd.read_csv(LAB_CSV) if os.path.exists(LAB_CSV) else pd.DataFrame()
for cid in DISTINCT:
    if len(lab) and cid in lab['config'].unique(): continue
    y, pr, paths = lab_predict(load_clf(cid, 'clf'), TEST_DF, CFG[cid]['clahe'], load_xgb(cid), desc=f'lab {cid}')
    d = pd.DataFrame({'config': cid, 'path': paths, 'y': y, 'pred': pr.argmax(1), 'conf': pr.max(1)})
    for k in range(NUM_CLASSES): d[f'p{k}'] = pr[:, k]
    lab = pd.concat([lab, d], ignore_index=True); lab.to_csv(LAB_CSV, index=False)
    print(f'{cid}: acc={(d.y == d.pred).mean():.4f}  errors={(d.y != d.pred).sum()}')
    if cid in ['A6', 'X4', 'Y4']: free_cache()

# Accuracy, ECE and Brier score per configuration
summ = []
for cid, d in lab.groupby('config', sort=False):
    P = d[[f'p{k}' for k in range(NUM_CLASSES)]].values
    summ.append(dict(config=cid, acc=(d.y == d.pred).mean(), errors=int((d.y != d.pred).sum()),
                     ece=ece_score(d.y, P), brier=brier(d.y, P)))
summ = pd.DataFrame(summ); display(summ.round(4)); summ.to_csv(f'{OUT_DIR}/B_lab_summary.csv', index=False)

# Confusion matrices (counts) and a figure for key configurations
cms = {}
for cid in DISTINCT:
    d = lab[lab.config == cid]; cms[cid] = confusion_matrix(d.y, d.pred, labels=range(NUM_CLASSES)).tolist()
save_json(cms, 'B_confusion_matrices.json')
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, cid in zip(axes, ['A1', 'X4', 'A8']):
    cm = np.array(cms[cid]); cmn = cm / cm.sum(1, keepdims=True)
    sns.heatmap(cmn, annot=np.array([[f'{cmn[i,j]:.3f}\n({cm[i,j]})' for j in range(4)] for i in range(4)]), fmt='',
                cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=ax, cbar=False, vmin=0, vmax=1)
    ax.set_title(cid); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/B_confusion_A1_X4_A8.png'); plt.show()

# Dominant confusion pairs
pairs = [('Unfermented', 'Broken'), ('Broken', 'Unfermented'), ('Fermented', 'Unfermented'), ('Unfermented', 'Fermented')]
rows = []
for cid in ['A1', 'X4', 'A8', 'Y4']:
    cm = np.array(cms[cid]); row = {'config': cid}
    for t, p in pairs: row[f'{t}->{p}'] = int(cm[class_to_idx[t], class_to_idx[p]])
    row['other'] = int(cm.sum() - np.trace(cm) - sum(row[f'{t}->{p}'] for t, p in pairs)); rows.append(row)
pairs_df = pd.DataFrame(rows); display(pairs_df); pairs_df.to_csv(f'{OUT_DIR}/B_confusion_pairs.csv', index=False)

# Exact McNemar tests on the same crops
piv = lab.assign(ok=lab.y == lab.pred).pivot_table(index='path', columns='config', values='ok', aggfunc='first')
mc = {}
for a, b in [('A1', 'A8'), ('X1', 'X4'), ('X4', 'Y4'), ('X4', 'X6'), ('A6', 'A8'), ('X1', 'Y1'), ('A1', 'X1'), ('X3', 'Y3')]:
    mc[f'{a}_vs_{b}'] = mcnemar(piv[a].values, piv[b].values)
mc_df = pd.DataFrame(mc).T; display(mc_df); save_json(mc, 'B_lab_mcnemar.json')
print('A1 vs X1: identical architecture trained twice -> estimate of run-to-run variation.')

## C. Unified field evaluation (white and sack backgrounds)
The inference protocol follows the earlier field script (resize to a 1920 canvas, score threshold 0.10, NMS 0.20, geometric filter, IoSA), with the following changes:
* drying-net images are excluded;
* all 20 configurations are evaluated with the same code;
* shared configurations are run in three variants: `replicate` (as in the earlier script: detector and classifier both use the detector's Swin weights), `final` (both use the classifier's Swin weights, i.e. the actual shared model after sequential training) and `split` (each uses its own Swin weights; diagnostic only);
* a **common detector** condition uses the X2 detector (standalone, DE anchors) for every classifier;
* OOD abstention is disabled for all runs (A8 with OOD is reported separately).

In [ ]:
# --- C0. Field ground truth (128 entries) ---
FIELD_GT = {'20260603_124627.jpg': {'classes': ['Moldy'], 'context': 'control_bg'}, '20260603_130320.jpg': {'classes': ['Moldy'], 'context': 'control_bg'}, '20260716_090019.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090034.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090017.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_085940.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090023.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090839.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090907.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090900.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_091601.jpg': {'classes': ['Unfermented', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091247.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_084332.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084315.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084317.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_083640.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083631.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083926.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_083836.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_083710.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083726.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_084351.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084328.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_083858.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_084454.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'sack_bg'}, '20260716_084511.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'sack_bg'}, '20260716_084450.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091757.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091334.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091400.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091031.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091008.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_090956.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091029.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_090405.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'control_bg'}, '20260716_090437.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'control_bg'}, '20260716_091356.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_090917.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090852.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_084521.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'sack_bg'}, '20260716_083724.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083651.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083728.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083658.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083635.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_083720.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_090028.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_090331.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'control_bg'}, '20260716_090346.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'control_bg'}, '20260716_090013.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090036.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090005.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090124.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_090118.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_090155.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_083825.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_083947.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_090139.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_090150.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_085951.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_085956.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090009.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090041.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_085932.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090909.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090902.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090929.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090920.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090842.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_091729.jpg': {'classes': ['Unfermented', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091443.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091643.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091743.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091656.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091619.jpg': {'classes': ['Unfermented', 'Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091821.jpg': {'classes': ['Unfermented', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_091242.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091329.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091228.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091002.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091224.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091257.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_084322.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084404.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084335.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_091316.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_090846.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_090914.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_083933.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'control_bg'}, '20260716_090105.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'control_bg'}, '20260716_090129.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_090135.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260716_083722.jpg': {'classes': ['Unfermented'], 'context': 'control_bg'}, '20260716_085943.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260716_090319.jpg': {'classes': ['Unfermented', 'Broken'], 'context': 'control_bg'}, '20260716_084506.jpg': {'classes': ['Unfermented', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091504.jpg': {'classes': ['Unfermented', 'Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091252.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260716_091636.jpg': {'classes': ['Broken', 'Unfermented', 'Broken'], 'context': 'sack_bg'}, '20260716_084640.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_084359.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_090850.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260716_084346.jpg': {'classes': ['Unfermented'], 'context': 'sack_bg'}, '20260716_091015.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260716_091235.jpg': {'classes': ['Broken', 'Unfermented'], 'context': 'sack_bg'}, '20260623_155639.jpg': {'classes': ['Fermented'], 'context': 'control_bg'}, '20260623_155755.jpg': {'classes': ['Fermented'], 'context': 'control_bg'}, '20260623_155719.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260623_155744.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260709_072641.jpg': {'classes': ['Broken'], 'context': 'control_bg'}, '20260602_202503.jpg': {'classes': ['Moldy', 'Moldy'], 'context': 'control_bg'}, '20260623_155704.jpg': {'classes': ['Broken', 'Broken'], 'context': 'control_bg'}, '20260709_072757.jpg': {'classes': ['Broken', 'Moldy'], 'context': 'control_bg'}, '20260603_130330.jpg': {'classes': ['Fermented', 'Moldy', 'Moldy'], 'context': 'control_bg'}, '20260623_155654.jpg': {'classes': ['Fermented', 'Broken', 'Broken'], 'context': 'control_bg'}, '20260627_103402.jpg': {'classes': ['Unfermented'], 'context': 'drying_net_bg'}, '20260627_103428.jpg': {'classes': ['Unfermented'], 'context': 'drying_net_bg'}, '20260709_072957.jpg': {'classes': ['Fermented'], 'context': 'sack_bg'}, '20260709_073029.jpg': {'classes': ['Fermented'], 'context': 'sack_bg'}, '20260709_073250.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260709_073255.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260709_073247.jpg': {'classes': ['Broken'], 'context': 'sack_bg'}, '20260709_073423.jpg': {'classes': ['Moldy'], 'context': 'sack_bg'}, '20260709_073430.jpg': {'classes': ['Moldy'], 'context': 'sack_bg'}, '20260709_073330.jpg': {'classes': ['Broken', 'Broken'], 'context': 'sack_bg'}, '20260709_073442.jpg': {'classes': ['Broken', 'Moldy'], 'context': 'sack_bg'}, '20260709_073141.jpg': {'classes': ['Fermented', 'Fermented'], 'context': 'sack_bg'}, '20260709_073516.jpg': {'classes': ['Broken', 'Moldy', 'Moldy'], 'context': 'sack_bg'}}
FIELD = {k: v for k, v in FIELD_GT.items() if v['context'] in ('control_bg', 'sack_bg')}
FIELD = {k: v for k, v in FIELD.items() if os.path.exists(f'{TEST_IMG_DIR}/{k}')}
print(f'Field images used: {len(FIELD)} (expected 126)', Counter(v['context'] for v in FIELD.values()))

In [ ]:
# --- C1. Field inference ---
OOD_T, OP_T, NMS_T, CANVAS = 0.60, 0.10, 0.20, 1920
IOSA_T, MAX_WF, MIN_AR, MAX_AR, MAX_AF = 0.40, 0.58, 0.35, 3.20, 0.42

def norm_entropy(p): return float(scipy_entropy(p) / np.log(len(p)))

def iosa(boxes, scores):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True); keep, sup = [], set()
    for i, oi in enumerate(order):
        if oi in sup: continue
        keep.append(oi); b1 = boxes[oi]; a1 = (b1[2] - b1[0]) * (b1[3] - b1[1])
        for oj in order[i + 1:]:
            if oj in sup: continue
            b2 = boxes[oj]; a2 = (b2[2] - b2[0]) * (b2[3] - b2[1])
            inter = max(0, min(b1[2], b2[2]) - max(b1[0], b2[0])) * max(0, min(b1[3], b2[3]) - max(b1[1], b2[1]))
            if inter > 0 and inter / max(min(a1, a2), 1) > IOSA_T: sup.add(oj)
    return [boxes[i] for i in keep], [scores[i] for i in keep]

def geom(boxes, scores, W, H):
    kb, ks = [], []
    for b, s in zip(boxes, scores):
        bw, bh = b[2] - b[0], b[3] - b[1]
        if bw / W > MAX_WF or not (MIN_AR <= bh / max(bw, 1) <= MAX_AR) or bw * bh / (W * H) > MAX_AF: continue
        kb.append(b); ks.append(s)
    return kb, ks

def detect(det, orig):
    W, H = orig.size; t = transforms.ToTensor()(orig.resize((CANVAS, CANVAS), Image.BILINEAR)).unsqueeze(0).to(DEVICE)
    with torch.no_grad(): p = det(t)[0]
    m = p['scores'].cpu() >= OP_T; b, s = p['boxes'].cpu()[m], p['scores'].cpu()[m]
    if len(b): k = tv_ops.nms(b, s, NMS_T); b, s = b[k], s[k]
    sx, sy = W / CANVAS, H / CANVAS
    bx = [[float(np.clip(x1 * sx, 0, W)), float(np.clip(y1 * sy, 0, H)), float(np.clip(x2 * sx, 0, W)), float(np.clip(y2 * sy, 0, H))]
          for x1, y1, x2, y2 in b.numpy()]
    bx, sc = geom(bx, [float(v) for v in s], W, H)
    if len(bx) > 1: bx, sc = iosa(bx, sc)
    return bx

def classify(clf, xgbm, clahe, crop):
    x = val_tf(apply_clahe_rgb(crop) if clahe else crop).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pr = xgbm.predict_proba(clf.get_latent_features(x).cpu().numpy())[0] if xgbm is not None else clf.get_softmax(x).cpu().numpy()[0]
    return pr

def match(pred, gt):
    pc, gc = Counter(pred), Counter(gt); tp = sum((pc & gc).values())
    return tp, sum(pc.values()) - tp, sum(gc.values()) - tp

# Runs: (run_id, cfg, det_source, det_swin, clf_swin)
RUNS = []
for cid in CFG:
    if CFG[cid]['shared']:
        RUNS += [(f'{cid}|replicate', cid, cid, 'det', 'det'), (f'{cid}|final', cid, cid, 'clf', 'clf'), (f'{cid}|split', cid, cid, 'det', 'clf')]
    else:
        RUNS += [(f'{cid}|native', cid, cid, 'det', 'clf')]
    RUNS += [(f'{cid}|commonX2', cid, 'X2', 'det', 'clf')]

FIELD_CSV = f'{OUT_DIR}/C_field_per_image.csv'
done = set(pd.read_csv(FIELD_CSV)['run'].unique()) if os.path.exists(FIELD_CSV) else set()
todo = [r for r in RUNS if r[0] not in done]
print(f'{len(RUNS)} runs, {len(todo)} pending')

# Group runs by detector so each image is detected once per detector
by_det = defaultdict(list)
for r in todo: by_det[(r[2], r[3])].append(r)
for (det_src, det_sw), runs in by_det.items():
    det = load_det(det_src, det_sw, score_thresh=OP_T); out = []
    for fn, g in tqdm(FIELD.items(), desc=f'det {det_src}/{det_sw} ({len(runs)} run)'):
        orig = ImageOps.exif_transpose(Image.open(f'{TEST_IMG_DIR}/{fn}')).convert('RGB')
        boxes = detect(det, orig); crops = [orig.crop(tuple(b)) for b in boxes]
        for run_id, cid, _, _, clf_sw in runs:
            clf = load_clf(cid, clf_sw); xg = load_xgb(cid); probs = [classify(clf, xg, CFG[cid]['clahe'], c) for c in crops]
            pred = [class_names[int(p.argmax())] for p in probs]
            pred_ood = [class_names[int(p.argmax())] for p in probs if norm_entropy(p) <= OOD_T]
            tp, fp, fn_ = match(pred, g['classes']); tpo, fpo, fno = match(pred_ood, g['classes'])
            out.append(dict(run=run_id, config=cid, variant=run_id.split('|')[1], file=fn, context=g['context'],
                            n_obj=len(g['classes']), gt='|'.join(g['classes']), pred='|'.join(pred), n_det=len(boxes),
                            tp=tp, fp=fp, fn=fn_, tp_ood=tpo, fp_ood=fpo, fn_ood=fno,
                            probs=json.dumps([[round(float(v), 5) for v in p] for p in probs])))
    df_new = pd.DataFrame(out)
    df_new.to_csv(FIELD_CSV, mode='a', header=not os.path.exists(FIELD_CSV), index=False)
    del det; free_cache()
print('Field inference complete ->', FIELD_CSV)

In [ ]:
# --- C2. Field F1 by background, McNemar, bootstrap, Spearman, calibration ---
FIELD_CSV = f'{OUT_DIR}/C_field_per_image.csv'   # redefined so that C2 can run without C1
fld = pd.read_csv(FIELD_CSV)
fld['stratum'] = fld['context'].map({'control_bg': 'white', 'sack_bg': 'sack'}) + np.where(fld.n_obj == 1, '_single', '_multi')
fld['correct'] = (fld.tp == fld.n_obj) & (fld.fp == 0)          # McNemar unit: image
STRATA = {'white_single': lambda d: d.stratum == 'white_single', 'white_multi': lambda d: d.stratum == 'white_multi',
          'sack_single': lambda d: d.stratum == 'sack_single', 'sack_multi': lambda d: d.stratum == 'sack_multi',
          'all_single': lambda d: d.n_obj == 1, 'all': lambda d: d.n_obj > 0}

def f1_table(sfx=''):
    rows = []
    for run, d in fld.groupby('run', sort=False):
        row = {'run': run}
        for s, f in STRATA.items():
            dd = d[f(d)]; row[s] = round(f1_from(dd[f'tp{sfx}'].sum(), dd[f'fp{sfx}'].sum(), dd[f'fn{sfx}'].sum())[0], 3)
        row['boxes_returned'] = int(d.n_det.sum()); rows.append(row)
    return pd.DataFrame(rows).set_index('run')
T13 = f1_table(); T13.to_csv(f'{OUT_DIR}/C_field_F1_by_stratum.csv'); display(T13)
T13o = f1_table('_ood'); T13o.to_csv(f'{OUT_DIR}/C_field_F1_by_stratum_with_OOD.csv')
print('A8 with OOD abstention:', T13o.loc['A8|replicate'].to_dict())

# Bootstrap 95% CI of single-bean F1 per background (2000 image resamples)
rng = np.random.default_rng(SEED); ci = {}
for run, d in fld.groupby('run', sort=False):
    ci[run] = {}
    for s in ['white_single', 'sack_single', 'all_single']:
        dd = d[STRATA[s](d)][['tp', 'fp', 'fn']].values; bs = []
        for _ in range(2000):
            t = dd[rng.integers(0, len(dd), len(dd))].sum(0); bs.append(f1_from(*t)[0])
        ci[run][s] = [round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3)]
save_json(ci, 'C_field_bootstrap_CI.json')

# Field McNemar (unit = single-bean image), per background
def field_mc(r1, r2, strata=('all_single', 'white_single', 'sack_single')):
    out = {}
    for s in strata:
        a = fld[(fld.run == r1) & STRATA[s](fld)].set_index('file').correct
        b = fld[(fld.run == r2) & STRATA[s](fld)].set_index('file').correct
        a, b = a.align(b, join='inner'); out[s] = mcnemar(a.values, b.values)
    return out
PAIRS = [(f'X{i}|native', f'Y{i}|replicate') for i in range(1, 7)] + [(f'X{i}|native', f'Y{i}|final') for i in range(1, 7)] + \
        [(f'X{i}|native', f'Y{i}|split') for i in range(1, 7)] + \
        [('X4|commonX2', 'Y4|commonX2'), ('X2|native', 'X4|native'), ('X1|native', 'X4|native'), ('X4|native', 'X6|native'),
         ('A2|native', 'A3|native'), ('A6|replicate', 'A8|replicate'), ('A6|split', 'A8|split'), ('A6|commonX2', 'A8|commonX2'),
         ('A3|native', 'X4|native'), ('A1|native', 'X1|native')]
MC = {f'{a} vs {b}': field_mc(a, b) for a, b in PAIRS}; save_json(MC, 'C_field_mcnemar.json')
display(pd.DataFrame({k: {s: f"b={v['b']} c={v['c']} p={v['p_exact']:.3g}" for s, v in d.items()} for k, d in MC.items()}).T)

# Spearman: lab accuracy vs field single-bean F1 (126 images)
lab_acc = pd.read_csv(f'{OUT_DIR}/B_lab_summary.csv').set_index('config')['acc'].to_dict()
lab_acc.update({'A2': lab_acc['A1'], 'X2': lab_acc['X1']})
sp = {}
for variant in ['replicate', 'final', 'split']:
    xs, ys = [], []
    for cid in CFG:
        run = f'{cid}|{variant}' if CFG[cid]['shared'] else f'{cid}|native'
        xs.append(lab_acc[cid]); ys.append(T13.loc[run, 'all_single'])
    r = spearmanr(xs, ys); sp[variant] = dict(rho=round(float(r.statistic), 3), p=round(float(r.pvalue), 3))
print('Spearman lab-vs-field:', sp); save_json(sp, 'C_spearman.json')

# Field calibration (strict: single-bean images with exactly one detection), per background
cal = {}
for run, d in fld.groupby('run', sort=False):
    cal[run] = {}
    for ctx in ['white', 'sack', 'all']:
        dd = d[(d.n_obj == 1) & (d.n_det == 1)]
        if ctx != 'all': dd = dd[dd.stratum.str.startswith(ctx)]
        if not len(dd): continue
        P = np.array([json.loads(p)[0] for p in dd['probs']]); y = np.array([class_to_idx[g] for g in dd['gt']])
        cal[run][ctx] = dict(n=len(y), acc=round(float((P.argmax(1) == y).mean()), 3), mean_conf=round(float(P.max(1).mean()), 3),
                             ece=round(ece_score(y, P), 4), brier=round(brier(y, P), 4), n_unique_conf=int(len(np.unique(P.max(1).round(5)))))
save_json(cal, 'C_field_calibration.json')
display(pd.DataFrame({k: v.get('all', {}) for k, v in cal.items()}).T)

## D. Anchor analysis: DE anchor values, standard vs DE IoU, convergence

In [ ]:
# --- D. Anchor analysis ---
sizes = np.array([[im['width'], im['height']] for im in coco['images']])
scale_factor = 224.0 / float(np.median(sizes[:, 0]))
gt = np.array([a['bbox'][2:4] for a in coco['annotations']]) * scale_factor
gt_cls = np.array([cat_map[a['category_id']] for a in coco['annotations']])
gw, gh = gt[:, 0], gt[:, 1]; base_size = float(np.sqrt(np.median(gw * gh)))

def best_iou(ratios, scales):
    aw = np.array([base_size * s * np.sqrt(r) for r in ratios for s in scales])[None]
    ah = np.array([base_size * s / np.sqrt(r) for r in ratios for s in scales])[None]
    inter = np.minimum(gw[:, None], aw) * np.minimum(gh[:, None], ah)
    return (inter / (gw[:, None] * gh[:, None] + aw * ah - inter + 1e-9)).max(1)

res_D = {'h_over_w_mean': float((gh / gw).mean()), 'h_over_w_sd': float((gh / gw).std()), 'base_size_px224': base_size,
         'note': 'Fitness computed on all annotations (train+val+test), as in the original anchor search.'}
for name, rat, sca in [('standard', STD_RATIOS, STD_SCALES), ('DE_chainA', acfg_A['opt_ratios'], acfg_A['opt_scales']),
                       ('DE_chainXY', acfg_XY['opt_ratios'], acfg_XY['opt_scales'])]:
    iou = best_iou(rat, sca)
    res_D[name] = dict(ratios=[round(r, 4) for r in rat], scales=[round(s, 4) for s in sca], n_anchors=len(rat) * len(sca),
                       mean_best_iou=round(float(iou.mean()), 4), frac_below_0_5=round(float((iou < 0.5).mean()), 4),
                       per_class_mean_iou={c: round(float(iou[gt_cls == c].mean()), 4) for c in class_names})

# Convergence curve: rerun DE with the same seed (should reproduce the stored JSON)
hist = []
def _fit(p):
    ratios, scales = p[:3], p[3:]
    return 1.0 - best_iou(ratios, scales).mean()
r = differential_evolution(_fit, bounds=[(0.2, 5.0)] * 3 + [(0.5, 3.0)] * 3, strategy='best1bin', maxiter=100, popsize=30,
                           seed=42, callback=lambda xk, convergence=None: hist.append(1 - _fit(xk)))
res_D['DE_rerun'] = dict(final_mean_best_iou_3ratios=round(1 - r.fun, 4), generations=len(hist), x=[round(v, 4) for v in r.x])
save_json(res_D, 'D_anchor_analysis.json'); print(json.dumps(res_D, indent=1))
plt.figure(figsize=(5.5, 3.5)); plt.plot(range(1, len(hist) + 1), hist, 'g-')
plt.axhline(res_D['standard']['mean_best_iou'], ls=':', c='gray', label='standard anchors')
plt.xlabel('Generation'); plt.ylabel('Mean best-anchor IoU'); plt.legend(); plt.title('DE anchor optimisation convergence')
plt.savefig(f'{OUT_DIR}/D_DE_convergence.png'); plt.show()

## E. Per-class AP (standard vs DE anchors) and bootstrap CI of ΔmAP

In [ ]:
# --- E. Per-class AP and bootstrap ---
N_BOOT = 100      # number of bootstrap resamples
res_E = load_json('E_perclass_ap.json') or {}
for std_id, de_id in [('A1', 'A2'), ('X1', 'X2'), ('Y1', 'Y2')]:
    key = f'{std_id}_vs_{de_id}'
    if key in res_E: print(key, 'already done -> skip'); continue
    P = {}
    for cid in [std_id, de_id]: P[cid] = det_predict_all(load_det(cid, 'det'), desc=f'det {cid}'); free_cache()
    out = {}
    for cid in [std_id, de_id]:
        m = map_of(*P[cid], per_class=True); m50 = map_of(*P[cid], per_class=True, iou=[0.5])
        cls_ids = m['classes'].tolist()
        out[cid] = dict(map5095=float(m['map']), map50=float(m['map_50']), map75=float(m['map_75']),
                        ap5095_per_class={class_names[c - 1]: round(float(v), 4) for c, v in zip(cls_ids, m['map_per_class'].tolist())},
                        ap50_per_class={class_names[c - 1]: round(float(v), 4) for c, v in zip(m50['classes'].tolist(), m50['map_per_class'].tolist())})
    rng = np.random.default_rng(SEED); n = len(P[std_id][0]); deltas = []
    for _ in tqdm(range(N_BOOT), desc=f'bootstrap {key}'):
        idx = rng.integers(0, n, n).tolist()
        deltas.append(float(map_of(*P[de_id], idx=idx)['map']) - float(map_of(*P[std_id], idx=idx)['map']))
    out['delta_map5095_pp'] = round((out[de_id]['map5095'] - out[std_id]['map5095']) * 100, 3)
    out['delta_ci95_pp'] = [round(float(np.percentile(deltas, 2.5)) * 100, 3), round(float(np.percentile(deltas, 97.5)) * 100, 3)]
    out['n_boot'] = N_BOOT; res_E[key] = out; save_json(res_E, 'E_perclass_ap.json'); print(key, json.dumps(out, indent=1))

## F. Computational efficiency

In [ ]:
# --- F. Efficiency: parameters, GFLOPs, per-stage latency, throughput ---
from torch.utils.flop_counter import FlopCounterMode
def n_params(m): return sum(p.numel() for p in m.parameters()) / 1e6
def gflops(m, x):
    try:
        with FlopCounterMode(display=False) as fc:
            with torch.no_grad(): m(x)
        return fc.get_total_flops() / 1e9
    except Exception as e:
        print('  FLOP counter failed:', e); return float('nan')

def timeit(fn, n=100, warm=10, dev=DEVICE):
    for _ in range(warm): fn()
    ts = []
    for _ in range(n):
        if dev.type == 'cuda': torch.cuda.synchronize()
        t0 = time.perf_counter(); fn()
        if dev.type == 'cuda': torch.cuda.synchronize()
        ts.append((time.perf_counter() - t0) * 1000)
    return round(float(np.mean(ts)), 2), round(float(np.std(ts)), 2)

res_F = {'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'cpu', 'complexity': {}, 'latency_gpu': {}, 'latency_cpu': {}, 'throughput': {}}
x224 = torch.randn(1, 3, 224, 224, device=DEVICE)
swin_params = n_params(_swin())
for cid in ['X1', 'X4', 'Y4', 'A8']:
    clf, det = load_clf(cid), load_det(cid); xg = load_xgb(cid)
    pc, pd_ = n_params(clf), n_params(det)
    total = pc + pd_ - (swin_params if CFG[cid]['shared'] else 0)
    res_F['complexity'][cid] = dict(params_clf_M=round(pc, 2), params_det_M=round(pd_, 2), params_total_M=round(total, 2),
                                    encoders=1 if CFG[cid]['shared'] else 2, gflops_clf=round(gflops(clf, x224), 2),
                                    gflops_det_224=round(gflops(det, [x224[0]]), 2), xgb_trees=(xg.n_estimators if hasattr(xg, 'n_estimators') else ('isotonic+xgb' if xg else None)))
display(pd.DataFrame(res_F['complexity']).T)

# Per-stage latency on real field images (first 20 white/sack images)
sample = list(FIELD)[:20]
def stage_latency(cid, dev, n_img=20, reps=3):
    clf = CLF_CLS[CFG[cid]['clf'][0]](NUM_CLASSES).to(dev); clf.load_state_dict(load_clf(cid).state_dict()); clf.eval()
    det = build_det(GENS[CFG[cid]['det'][1]], OP_T).to(dev); det.load_state_dict(load_det(cid).state_dict()); det.eval()
    xg = load_xgb(cid); T = defaultdict(list)
    def sync():
        if dev.type == 'cuda': torch.cuda.synchronize()
    for fn in sample[:n_img]:
        for _ in range(reps):
            t0 = time.perf_counter(); orig = ImageOps.exif_transpose(Image.open(f'{TEST_IMG_DIR}/{fn}')).convert('RGB')
            t = transforms.ToTensor()(orig.resize((CANVAS, CANVAS), Image.BILINEAR)).unsqueeze(0).to(dev); sync(); t1 = time.perf_counter()
            with torch.no_grad(): p = det(t)[0]
            sync(); t2 = time.perf_counter()
            W, H = orig.size; m = p['scores'].cpu() >= OP_T; b = p['boxes'].cpu()[m]
            crops = [orig.crop((float(x1) * W / CANVAS, float(y1) * H / CANVAS, float(x2) * W / CANVAS, float(y2) * H / CANVAS)) for x1, y1, x2, y2 in b[:3]] or [orig]
            t3 = time.perf_counter(); tc = tx = 0.0
            for c in crops:
                a = time.perf_counter(); xx = val_tf(apply_clahe_rgb(c) if CFG[cid]['clahe'] else c).unsqueeze(0).to(dev)
                with torch.no_grad():
                    if xg is not None:
                        f = clf.get_latent_features(xx).cpu().numpy(); sync(); bb = time.perf_counter(); xg.predict_proba(f); tx += time.perf_counter() - bb
                    else: clf.get_softmax(xx).cpu()
                sync(); tc += time.perf_counter() - a
            T['preprocess_ms'].append((t1 - t0) * 1000); T['detection_ms'].append((t2 - t1) * 1000)
            T['postproc_crop_ms'].append((t3 - t2) * 1000); T['classify_per_bean_ms'].append((tc - tx) * 1000 / len(crops))
            T['xgb_per_bean_ms'].append(tx * 1000 / len(crops))
            T['total_ms'].append((t3 - t0 + tc) * 1000)
    del clf, det; free_cache()
    return {k: [round(float(np.mean(v[reps:])), 2), round(float(np.std(v[reps:])), 2)] for k, v in T.items()}
for cid in ['X4', 'Y4', 'A8']:
    res_F['latency_gpu'][cid] = stage_latency(cid, DEVICE)
    res_F['latency_cpu'][cid] = stage_latency(cid, torch.device('cpu'), n_img=5, reps=2)
display(pd.DataFrame({f'{c}_gpu': v for c, v in res_F['latency_gpu'].items()} | {f'{c}_cpu': v for c, v in res_F['latency_cpu'].items()}))

# Throughput: X4 classifier (crops/s) and X2 detector (images/s, 224 input)
for dev in [DEVICE, torch.device('cpu')]:
    clf = AdaptiveSwinCNN_EMDMFM(NUM_CLASSES).to(dev); clf.load_state_dict(load_clf('X4').state_dict()); clf.eval()
    det = build_det(DE_GEN_XY, OP_T).to(dev); det.load_state_dict(load_det('X2').state_dict()); det.eval()
    for bs in [1, 8, 32]:
        xb = torch.randn(bs, 3, 224, 224, device=dev); n = 30 if dev.type == 'cuda' else 5
        with torch.no_grad():
            mc, _ = timeit(lambda: clf(xb), n=n, warm=3, dev=dev); md_, _ = timeit(lambda: det(list(xb)), n=n, warm=3, dev=dev)
        res_F['throughput'][f'{dev.type}_bs{bs}'] = dict(clf_crops_per_s=round(bs / mc * 1000, 1), det_imgs_per_s=round(bs / md_ * 1000, 1))
    del clf, det; free_cache()
display(pd.DataFrame(res_F['throughput']).T); save_json(res_F, 'F_efficiency.json')

## G. Grad-CAM: X4 vs Y4

In [ ]:
# --- G. Grad-CAM on the last extracted Swin stage (no extra library) ---
class SwinGradCAM:
    def __init__(self, model):
        self.m = model; self.act = None
        model.swin.register_forward_hook(self._hook)
    def _hook(self, mod, inp, out):
        a = out[2]
        if a.requires_grad: a.retain_grad()
        self.act = a
    def __call__(self, x, cls=None):
        self.m.zero_grad(); x = x.clone().requires_grad_(True)
        logits = self.m(x); cls = int(logits.argmax(1)) if cls is None else cls
        logits[0, cls].backward()
        A = _bchw(self.act); G = _bchw(self.act.grad)
        cam = F.relu((G.mean((2, 3), keepdim=True) * A).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=(224, 224), mode='bilinear', align_corners=False)[0, 0].detach().cpu().numpy()
        return (cam - cam.min()) / (cam.max() - cam.min() + 1e-8), cls, float(F.softmax(logits, 1)[0, cls])

def overlay(pil, cam):
    img = np.array(pil.resize((224, 224))) / 255.0; hm = plt.cm.jet(cam)[..., :3]
    return np.clip(0.55 * img + 0.45 * hm, 0, 1)

models_g = {'X4 (decoupled)': (load_clf('X4', 'clf'), False), 'Y4 (shared, clf Swin)': (load_clf('Y4', 'clf'), False),
            'Y4 (shared, det Swin = old field run)': (load_clf('Y4', 'det'), False)}
cams = {k: SwinGradCAM(copy.deepcopy(m)) for k, (m, _) in models_g.items()}
# Lab: one crop per class; field: two white + two sack single-bean images, cropped by the X2 detector
lab_imgs = [(Image.open(TEST_DF[TEST_DF.label == k].iloc[3].image_path).convert('RGB'), f'lab {class_names[k]}') for k in range(4)]
det_x2 = load_det('X2', 'det', OP_T); field_imgs = []
for ctx in ['control_bg', 'sack_bg']:
    for fn in [f for f, g in FIELD.items() if g['context'] == ctx and len(g['classes']) == 1][:2]:
        orig = ImageOps.exif_transpose(Image.open(f'{TEST_IMG_DIR}/{fn}')).convert('RGB'); bx = detect(det_x2, orig)
        if bx: field_imgs.append((orig.crop(tuple(bx[0])), f"{'white' if ctx == 'control_bg' else 'sack'} GT={FIELD[fn]['classes'][0]}"))
items = lab_imgs + field_imgs
fig, axes = plt.subplots(len(cams) + 1, len(items), figsize=(2.3 * len(items), 2.4 * (len(cams) + 1)))
for j, (im, lbl) in enumerate(items):
    axes[0, j].imshow(im.resize((224, 224))); axes[0, j].set_title(lbl, fontsize=8); axes[0, j].axis('off')
    for i, (k, cam) in enumerate(cams.items(), 1):
        x = val_tf(im).unsqueeze(0).to(DEVICE); c, cls, p = cam(x)
        axes[i, j].imshow(overlay(im, c)); axes[i, j].set_title(f'{class_names[cls]} {p:.2f}', fontsize=8); axes[i, j].axis('off')
        if j == 0: axes[i, j].text(-30, 112, k, rotation=90, va='center', ha='right', fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT_DIR}/G_gradcam_X4_Y4.png'); plt.show()

## H. Dataset sufficiency (learning curve)
Retrains the X1 (linear head) and X4 (AdaptiveSwin-CNN) classifiers — both standalone, without CLAHE, using the same recipe as the original X runs — on 10/25/50/75% of the training data (class-stratified subsampling at the source-image level). The 100% points are taken from the existing results. Training resumes from per-epoch checkpoints.

In [ ]:
# --- H/I. Training function (same recipe as the ablation notebooks) ---
CLF_EP, BATCH = 15, 16
TRAIN_CKPT_DIR = f'{OUT_DIR}/ckpt'; os.makedirs(TRAIN_CKPT_DIR, exist_ok=True)

def train_eval_classifier(arch, tr_df, va_df, te_df, ckpt_path):
    set_seed(); model = CLF_CLS[arch](NUM_CLASSES).to(DEVICE)
    tr_ld = DataLoader(ClfDataset(tr_df, train_tf), batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
    va_ld = loader(va_df); te_ld = loader(te_df)
    labels = tr_df.label.values
    cw = torch.tensor(compute_class_weight('balanced', classes=np.unique(labels), y=labels), dtype=torch.float).to(DEVICE)
    crit = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    sch = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CLF_EP, eta_min=1e-6)
    scaler = torch.amp.GradScaler('cuda'); start, best, best_w, hist = 0, 0.0, None, []
    if os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location=DEVICE); model.load_state_dict(ck['model_state'])
        opt.load_state_dict(ck['opt']); sch.load_state_dict(ck['sch']); start, best, hist = ck['epoch'] + 1, ck['best'], ck['hist']
        best_w = ck['best_w']; print(f'  [resume] epoch {start}/{CLF_EP}')
    for ep in range(start, CLF_EP):
        model.train(); c = t = 0
        for x, y, _ in tqdm(tr_ld, desc=f'  ep {ep + 1}/{CLF_EP}', leave=False):
            x, y = x.to(DEVICE), y.to(DEVICE); opt.zero_grad()
            with torch.amp.autocast('cuda'): out = model(x); loss = crit(out, y)
            scaler.scale(loss).backward(); scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            scaler.step(opt); scaler.update(); c += (out.argmax(1) == y).sum().item(); t += len(y)
        sch.step(); model.eval(); vc = vt = 0
        with torch.no_grad():
            for x, y, _ in va_ld: vc += (model(x.to(DEVICE)).argmax(1).cpu() == y).sum().item(); vt += len(y)
        tr_acc, va_acc = c / t, vc / vt; hist.append([tr_acc, va_acc])
        if va_acc > best: best = va_acc; best_w = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({'epoch': ep, 'model_state': model.state_dict(), 'opt': opt.state_dict(), 'sch': sch.state_dict(),
                    'best': best, 'best_w': best_w, 'hist': hist}, ckpt_path)
        print(f'  ep {ep + 1:02d} train={tr_acc:.4f} val={va_acc:.4f}')
    model.load_state_dict(best_w); model.eval(); ys, ps = [], []
    with torch.no_grad():
        for x, y, _ in te_ld: ps.append(model.get_softmax(x.to(DEVICE)).cpu().numpy()); ys.extend(y.numpy())
    P = np.vstack(ps); ys = np.array(ys)
    rec = classification_report(ys, P.argmax(1), target_names=class_names, output_dict=True, digits=4)
    return model, dict(test_acc=float((P.argmax(1) == ys).mean()), macro_f1=float(f1_score(ys, P.argmax(1), average='macro')),
                       best_val=float(best), final_train_val_gap_pp=round((hist[-1][0] - hist[-1][1]) * 100, 3),
                       recall={c: round(rec[c]['recall'], 4) for c in class_names}, hist=hist, n_train=len(tr_df))

def group_subsample(df, frac, seed=SEED):
    keep = []
    for k, d in df.groupby('label'):
        groups = d.group.unique(); rng = np.random.default_rng(seed + k)
        sel = set(rng.choice(groups, max(1, int(round(frac * len(groups)))), replace=False)); keep.append(d[d.group.isin(sel)])
    return pd.concat(keep).reset_index(drop=True)

In [ ]:
# --- H. Dataset-sufficiency learning curve ---
res_H = load_json('H_sufficiency.json') or {}
for arch, name in [('basic', 'X1'), ('adaptive', 'X4')]:
    for frac in [0.10, 0.25, 0.50, 0.75]:
        key = f'{name}_{int(frac * 100)}'
        if key in res_H: print(key, 'already done'); continue
        print(f'\n=== {key} ===')
        _, r = train_eval_classifier(arch, group_subsample(TRAIN_DF, frac), VAL_DF, TEST_DF, f'{TRAIN_CKPT_DIR}/H_{key}.pth')
        res_H[key] = r; save_json(res_H, 'H_sufficiency.json'); free_cache()
res_H.setdefault('X1_100', {'test_acc': 0.9929, 'n_train': len(TRAIN_DF), 'note': 'from the X1-X6 notebook'})
res_H.setdefault('X4_100', {'test_acc': 0.9913, 'n_train': len(TRAIN_DF), 'note': 'from the X1-X6 notebook'})
save_json(res_H, 'H_sufficiency.json')
fr = [10, 25, 50, 75, 100]
plt.figure(figsize=(5.5, 3.6))
for name, mk in [('X1', 'o-'), ('X4', 's-')]:
    plt.plot([res_H[f'{name}_{f}']['n_train'] for f in fr], [res_H[f'{name}_{f}']['test_acc'] * 100 for f in fr], mk, label=name)
plt.xlabel('Training crops'); plt.ylabel('Test accuracy (%)'); plt.legend(); plt.grid(alpha=.3)
plt.savefig(f'{OUT_DIR}/H_learning_curve.png'); plt.show()
display(pd.DataFrame({k: {kk: v.get(kk) for kk in ['n_train', 'test_acc', 'macro_f1', 'final_train_val_gap_pp']} for k, v in res_H.items()}).T)

## I. 5-fold cross-validation
All 16,012 crops are pooled and split with `StratifiedGroupKFold` (group = source image, so crops from the same image never fall in different folds). In each fold, 15% of the training groups are held out for validation. Models: **X1** (linear head), **X4** (AdaptiveSwin-CNN) and **X6** (XGBoost + isotonic calibration on the X4 embeddings of the same fold).

A8 is not included because a shared classifier must be trained after its detector (about 7–9 h per fold); X6 applies the same post-hoc stage on top of a separate encoder.

In [ ]:
# --- I. 5-fold cross-validation ---
ALL_DF = pd.concat([TRAIN_DF, VAL_DF, TEST_DF], ignore_index=True)
skf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
res_I = load_json('I_cv5.json') or {}
for k, (tr_idx, te_idx) in enumerate(skf.split(ALL_DF, ALL_DF.label, ALL_DF.group)):
    trv, te = ALL_DF.iloc[tr_idx].reset_index(drop=True), ALL_DF.iloc[te_idx].reset_index(drop=True)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
    a, b = next(gss.split(trv, trv.label, trv.group)); tr, va = trv.iloc[a], trv.iloc[b]
    for arch, name in [('basic', 'X1'), ('adaptive', 'X4')]:
        key = f'{name}_fold{k}'
        if key in res_I: print(key, 'already done'); continue
        print(f'\n=== {key}: train={len(tr)} val={len(va)} test={len(te)} ===')
        model, r = train_eval_classifier(arch, tr, va, te, f'{TRAIN_CKPT_DIR}/I_{key}.pth'); r.pop('hist')
        if name == 'X4' and f'X6_fold{k}' not in res_I:
            def emb(df):
                E, Y = [], []
                with torch.no_grad():
                    for x, y, _ in loader(df): E.append(model.get_latent_features(x.to(DEVICE)).cpu().numpy()); Y.extend(y.numpy())
                return np.vstack(E), np.array(Y)
            (Xtr, ytr), (Xva, yva), (Xte, yte) = emb(tr), emb(va), emb(te)
            xg = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05, objective='multi:softprob', num_class=NUM_CLASSES,
                                   tree_method='hist', device='cuda', random_state=SEED).fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            cal = CalibratedClassifierCV(xg, method='isotonic', cv='prefit').fit(Xva, yva); P = cal.predict_proba(Xte)
            res_I[f'X6_fold{k}'] = dict(test_acc=float((P.argmax(1) == yte).mean()), macro_f1=float(f1_score(yte, P.argmax(1), average='macro')),
                                        recall={c: round(float(((P.argmax(1) == yte) & (yte == i)).sum() / (yte == i).sum()), 4) for i, c in enumerate(class_names)},
                                        ece=ece_score(yte, P))
        res_I[key] = r; save_json(res_I, 'I_cv5.json'); del model; free_cache()

cv = {}
for name in ['X1', 'X4', 'X6']:
    rs = [res_I[f'{name}_fold{k}'] for k in range(5) if f'{name}_fold{k}' in res_I]
    if not rs: continue
    f = lambda v: f'{np.mean(v) * 100:.2f} ± {np.std(v, ddof=1) * 100:.2f}' if len(v) > 1 else f'{v[0] * 100:.2f}'
    cv[name] = {'folds': len(rs), 'accuracy': f([r['test_acc'] for r in rs]), 'macro_f1': f([r['macro_f1'] for r in rs]),
                'recall_Broken': f([r['recall']['Broken'] for r in rs]), 'recall_Unfermented': f([r['recall']['Unfermented'] for r in rs])}
display(pd.DataFrame(cv).T); save_json(cv, 'I_cv5_summary.json')

## J. Summary of output files

In [ ]:
# --- J. Summary ---
for f in sorted(os.listdir(OUT_DIR)):
    p = f'{OUT_DIR}/{f}'
    if os.path.isfile(p): print(f'{f:45s} {os.path.getsize(p) / 1024:8.1f} KB')
print('\nDownload the full output folder:')
!cd "{BASE_DIR}" && zip -qr /content/bab4_completion.zip bab4_completion -x "bab4_completion/ckpt/*"
from google.colab import files; files.download('/content/bab4_completion.zip')